# Week 2: Dataset Structuring and Validation

This notebook focuses on validating the residential listing and sold datasets generated in Week 1.

Main objectives:

- Review dataset structure and schema
- Compare listing and sold fields
- Identify duplicate columns
- Analyze missing values
- Flag high-missing columns
- Check duplicate records
- Generate numeric summaries
- Conduct percentile analysis
- Visualize key distributions
## Part 1: Load Combined Residential Datasets

In this section, we load the aggregated residential datasets from Week 1 and inspect their basic structure.

## Part 2: Dataset Preview

Preview the first few rows to understand how the data is structured.

In [ ]:
# --------------------------------------------------
# Week 2: Dataset Structuring and Validation
# Part 1: Load combined datasets and inspect structure
# --------------------------------------------------

import pandas as pd

# Load Week 1 output files directly
listings = pd.read_csv(
    "../outputs/combined_listings_residential.csv",
    low_memory=False
)

sold = pd.read_csv(
    "../outputs/combined_sold_residential.csv",
    low_memory=False
)

# Check dataset shape
print("Listings shape:", listings.shape)
print("Sold shape:", sold.shape)

# Preview first 5 rows
print("\nListings preview:")
print(listings.head())

print("\nSold preview:")
print(sold.head())

# Check all column names
print("\nListing columns:")
print(listings.columns.tolist())

print("\nSold columns:")
print(sold.columns.tolist())

# Check data types
print("\nListing data types:")
print(listings.dtypes)

print("\nSold data types:")
print(sold.dtypes)

## Part 3: Schema Comparison

Compare the column structures between listing and sold datasets to identify differences.

In [ ]:
# Find columns only in listings
listing_only = set(listings.columns) - set(sold.columns)

# Find columns only in sold
sold_only = set(sold.columns) - set(listings.columns)

print("Columns only in Listings:")
print(listing_only)

print("\nColumns only in Sold:")
print(sold_only)


The schema comparison shows that the listing and sold datasets do not share identical structures.

Several listing-only columns appeared with a `.1` suffix, which suggests duplicated fields caused by certain `_filled` source files.

The sold dataset also contains several unique property-feature fields such as `BasementYN`, `WaterfrontYN`, and `PoolPrivateYN`, which may be useful for future segmentation analysis.

This step helps identify structural inconsistencies before further validation.

## Part 4: Duplicate Column Cleanup

Some listing files contain duplicate columns with ".1" suffixes. These are removed to maintain schema consistency.

In [ ]:
# Find duplicate ".1" columns
duplicate_cols = [col for col in listings.columns if col.endswith('.1')]

print("Duplicate columns:")
print(duplicate_cols)

# Drop duplicate columns
listings = listings.drop(columns=duplicate_cols)

print("New Listings shape:", listings.shape)



Duplicate columns with `.1` suffixes were removed from the listings dataset.

These duplicated fields were not meaningful additional variables, but repeated versions of existing columns.

Removing them improves schema consistency and prevents redundancy during future analysis.

## Part 5: Missing Value Analysis

Analyze missing values across all columns to evaluate data completeness.

In [ ]:
# --------------------------------------------------
# Part 2: Missing Value Analysis
# --------------------------------------------------

# Count missing values
listing_missing_count = listings.isnull().sum()
sold_missing_count = sold.isnull().sum()

# Calculate missing percentages
listing_missing_pct = listings.isnull().mean() * 100
sold_missing_pct = sold.isnull().mean() * 100

# Create summary tables
listing_missing_summary = pd.DataFrame({
    "Missing Count": listing_missing_count,
    "Missing Percent": listing_missing_pct
}).sort_values("Missing Percent", ascending=False)

sold_missing_summary = pd.DataFrame({
    "Missing Count": sold_missing_count,
    "Missing Percent": sold_missing_pct
}).sort_values("Missing Percent", ascending=False)

# Top 20 missing columns
print("Top 20 Listing Missing Columns:")
print(listing_missing_summary.head(20))

print("\nTop 20 Sold Missing Columns:")
print(sold_missing_summary.head(20))



The missing value analysis shows that several fields have substantial incompleteness.

Most of the highest-missing fields are related to tax information, school districts, and optional property features.

This suggests that these variables may have limited analytical value or may require special handling during cleaning.

## Part 6: High Missing Fields (>90%)

Flag columns with more than 90% missing values for possible exclusion during data cleaning.

In [ ]:
# --------------------------------------------------
# Part 3: Flag high-missing columns (>90%)
# --------------------------------------------------

# Listing columns with >90% missing
listing_high_missing = listing_missing_summary[
    listing_missing_summary["Missing Percent"] > 90
]

# Sold columns with >90% missing
sold_high_missing = sold_missing_summary[
    sold_missing_summary["Missing Percent"] > 90
]

print("Listing columns >90% missing:")
print(listing_high_missing)

print("\nSold columns >90% missing:")
print(sold_high_missing)



Fields with more than 90% missing values were flagged for review.

These variables are strong candidates for exclusion in later cleaning stages because they provide very limited usable information.

However, they are not removed at this stage in order to preserve the raw dataset structure.

In [ ]:
# Drop columns with >90% missing values
listing_drop_cols = listing_missing_summary[
    listing_missing_summary["Missing Percent"] > 90
].index

sold_drop_cols = sold_missing_summary[
    sold_missing_summary["Missing Percent"] > 90
].index

listings = listings.drop(columns=listing_drop_cols)
sold = sold.drop(columns=sold_drop_cols)

print("Listings shape after dropping high-missing columns:", listings.shape)
print("Sold shape after dropping high-missing columns:", sold.shape)


Following the updated project guideline, columns with more than 90% missing values were removed from the working dataset because they are unlikely to contribute meaningful information to the final analysis.

## Part 7: Duplicate Record Analysis

Check for fully duplicated rows in both datasets.

In [ ]:
listing_duplicates = listings.duplicated().sum()
sold_duplicates = sold.duplicated().sum()

print("Listing duplicated rows:", listing_duplicates)
print("Sold duplicated rows:", sold_duplicates)


Fields with more than 90% missing values were flagged for review.

These variables are strong candidates for exclusion in later cleaning stages because they provide very limited usable information.

However, they are not removed at this stage in order to preserve the raw dataset structure.

## Part 8: Duplicate Sold Record Inspection

Preview duplicated sold rows for further review.

In [ ]:
# Show duplicate sold records
sold_duplicates_rows = sold[sold.duplicated()]

print(sold_duplicates_rows.head(10))
print("Duplicate rows shape:", sold_duplicates_rows.shape)


Inspection of duplicated sold records confirms that these rows are full-row duplicates rather than partial overlaps.

This suggests that duplicate removal can likely be performed safely in later cleaning stages.

## Part 9: Core Numeric Summary

Generate summary statistics for key market variables.

In [ ]:


numeric_cols = [
    'ClosePrice',
    'OriginalListPrice',
    'LivingArea',
    'DaysOnMarket',
    'BedroomsTotal',
    'BathroomsTotalInteger'
]

print(sold[numeric_cols].describe())


The numeric summary reveals several data quality issues.

Close price and original list price both contain zero values and extremely large outliers.

Living area also includes unrealistic maximum values, while Days on Market contains negative values.

These findings indicate the need for outlier handling and validation in future cleaning stages.

## Part 10: Percentile Analysis

Use percentile summaries to identify extreme outliers.

In [ ]:


for col in numeric_cols:
    print(f"\nPercentiles for {col}:")
    print(
        sold[col].describe(
            percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
        )
    )



Percentile analysis provides a clearer view of the normal market range.

For example, the 99th percentile of close price is far lower than the maximum value, confirming the presence of extreme outliers.

This step helps distinguish typical market behavior from abnormal records.

## Part 11: Distribution Visualization

Visualize the typical distributions of key variables using the 1st to 99th percentile range.

In [ ]:
import matplotlib.pyplot as plt

# ClosePrice: zoom in to 1% - 99%
close_filtered = sold[
    (sold["ClosePrice"] >= sold["ClosePrice"].quantile(0.01)) &
    (sold["ClosePrice"] <= sold["ClosePrice"].quantile(0.99))
]

close_filtered["ClosePrice"].hist(bins=50)
plt.title("Close Price Distribution (1st to 99th Percentile)")
plt.xlabel("Close Price")
plt.ylabel("Frequency")
plt.show()


# LivingArea: zoom in to 1% - 99%
area_filtered = sold[
    (sold["LivingArea"] >= sold["LivingArea"].quantile(0.01)) &
    (sold["LivingArea"] <= sold["LivingArea"].quantile(0.99))
]

area_filtered["LivingArea"].hist(bins=50)
plt.title("Living Area Distribution (1st to 99th Percentile)")
plt.xlabel("Living Area")
plt.ylabel("Frequency")
plt.show()


# DaysOnMarket: zoom in to 1% - 99%
dom_filtered = sold[
    (sold["DaysOnMarket"] >= sold["DaysOnMarket"].quantile(0.01)) &
    (sold["DaysOnMarket"] <= sold["DaysOnMarket"].quantile(0.99))
]

dom_filtered["DaysOnMarket"].hist(bins=50)
plt.title("Days on Market Distribution (1st to 99th Percentile)")
plt.xlabel("Days on Market")
plt.ylabel("Frequency")
plt.show()


Percentile analysis provides a clearer view of the normal market range.

For example, the 99th percentile of close price is far lower than the maximum value, confirming the presence of extreme outliers.

This step helps distinguish typical market behavior from abnormal records.